# Recursive Resolver Torture Test — "Silver-Bullet" Packet Hunt

**Goal:** find query packets (and malformed *upstream* responses) that crash or
wedge a recursive DNS resolver sitting between us (the stub) and the
authoritative servers.

```
  [ this notebook ]  --queries-->  [ RECURSIVE RESOLVER ]  --queries-->  [ our malicious AUTH server ]
   stub / fuzzer         (you run & watch this)              (this notebook, Part B)
```

This notebook has three parts:

* **Part A — Query-side fuzzer.** Hand-crafts every permutation of query packet
  (qtypes, opcodes/flags/rcodes, EDNS0 options, EDNS Client Subnet, QNAME
  length/label/depth boundaries, count mismatches, truncation) and fires them at
  the resolver, canary-checking resolver health after each packet to isolate the
  exact killer.
* **Part B — Malicious authoritative server.** A pure-Python UDP+TCP server that
  answers the resolver's *upstream* queries with RFC-violating responses (bad
  compression pointers, count lies, oversized/illegal names, CNAME loops, huge
  RRsets, malformed EDNS, bad RDLENGTH, truncation tricks…). Many resolver
  crashes live in response-parsing, not query-parsing.
* **Part C — End-to-end hunt.** Drives normal queries *through* the resolver for
  names in our test zone, forcing it to fetch and parse the malicious responses,
  and watches resolver health.

### Ground rules baked in
* **No internet.** A guard refuses to send to anything that isn't loopback or
  RFC-1918/ULA/link-local. Point everything at your lab.
* **You run the resolver.** Configure it to delegate/forward the test zone to
  this notebook's auth server (Part B prints the exact stanza).
* Everything is pure Python + `dnspython`; no root, no scapy, no raw sockets.

> ⚠️ Only run this against resolvers **you own and are authorized to test**.
> Crash-testing DNS software you don't control is an attack, not a test.

## 0. Configuration & safety guard

Set your lab targets here. The guard blocks any public IP so nothing can leak to
the real internet.

In [ ]:
import ipaddress, socket, struct, random, time, threading, itertools
from dataclasses import dataclass, field

# ── Targets (EDIT THESE) ────────────────────────────────────────────────────
RESOLVER_IP   = "127.0.0.1"   # the recursive resolver under test (you run it)
RESOLVER_PORT = 53

# The malicious authoritative server we stand up in Part B. It binds inside the
# container/kernel. Advertise an address the resolver can actually reach.
AUTH_BIND_IP  = "0.0.0.0"     # what the Python server listens on
AUTH_PORT     = 5354          # >1024 so we don't need root; point resolver here
AUTH_ADVERTISED_IP = "127.0.0.1"   # address the RESOLVER should use to reach us

# Zone we pretend to be authoritative for. The resolver must delegate/forward
# this to AUTH_ADVERTISED_IP:AUTH_PORT (Part B prints the config).
TEST_ZONE = "fuzz.test."

# A name the resolver can always answer quickly, used as the liveness canary.
# During Part C we use a name inside TEST_ZONE that our auth server answers
# cleanly; for Part A any cheap name works.
CANARY_NAME = "canary." + TEST_ZONE

# ── Safety knobs ────────────────────────────────────────────────────────────
ALLOW_PUBLIC_TARGET = False   # keep False. Set True only with written authz.
DEFAULT_UDP_TIMEOUT = 2.0
DEFAULT_TCP_TIMEOUT = 3.0

def _is_lab_ip(ip: str) -> bool:
    try:
        a = ipaddress.ip_address(ip)
    except ValueError:
        return False
    return a.is_loopback or a.is_private or a.is_link_local

def guard_target(ip: str):
    """Refuse to send anywhere that isn't a lab address unless explicitly allowed."""
    if ALLOW_PUBLIC_TARGET:
        return
    if not _is_lab_ip(ip):
        raise RuntimeError(
            f"BLOCKED: {ip!r} is not loopback/private/link-local. "
            f"This notebook must not touch the internet. "
            f"Set ALLOW_PUBLIC_TARGET=True only if you are authorized.")

guard_target(RESOLVER_IP)
print(f"Resolver under test : {RESOLVER_IP}:{RESOLVER_PORT}")
print(f"Malicious auth server: bind {AUTH_BIND_IP}:{AUTH_PORT}  (advertise {AUTH_ADVERTISED_IP})")
print(f"Test zone            : {TEST_ZONE}")
print("Safety guard         : ACTIVE (lab addresses only)" if not ALLOW_PUBLIC_TARGET else "Safety guard: DISABLED")

## 1. Wire-format DNS encoder (full byte-level control)

`dnspython` won't build *illegal* packets, so we hand-assemble bytes. These
primitives let us set any header bit, lie about section counts, forge EDNS
options, and construct QNAMEs that violate every length/label/pointer rule.

In [ ]:
# ── Header ──────────────────────────────────────────────────────────────────
def build_header(txid, *, qr=0, opcode=0, aa=0, tc=0, rd=1, ra=0, z=0, ad=0,
                 cd=0, rcode=0, qd=1, an=0, ns=0, ar=0):
    flags = ((qr & 1) << 15) | ((opcode & 0xF) << 11) | ((aa & 1) << 10) \
          | ((tc & 1) << 9) | ((rd & 1) << 8) | ((ra & 1) << 7) \
          | ((z & 1) << 6) | ((ad & 1) << 5) | ((cd & 1) << 4) | (rcode & 0xF)
    return struct.pack("!HHHHHH", txid & 0xFFFF, flags & 0xFFFF,
                       qd & 0xFFFF, an & 0xFFFF, ns & 0xFFFF, ar & 0xFFFF)

# ── Names ───────────────────────────────────────────────────────────────────
def enc_name(name: str, *, encoding="idna") -> bytes:
    """Well-formed name encoder. `encoding='latin-1'` passes bytes through raw."""
    if name in ("", "."):
        return b"\x00"
    out = b""
    for label in name.rstrip(".").split("."):
        lb = label.encode("latin-1") if encoding == "latin-1" else label.encode("idna") if label else b""
        if len(lb) > 63:
            raise ValueError("label > 63 (use raw_name for illegal lengths)")
        out += bytes([len(lb)]) + lb
    return out + b"\x00"

def raw_name(parts) -> bytes:
    """Assemble a QNAME from explicit pieces for illegal constructions.

    parts is a list of tuples:
      ("label", b"www")          normal label (len auto, masked to 6 bits)
      ("len",   n, b"payload")   arbitrary length byte n (allows >63 == illegal)
      ("ptr",   offset)          compression pointer 0xC000|offset (illegal in Q)
      ("raw",   b"...")          raw bytes spliced in verbatim
    Terminates with a root label unless a ptr/raw already ends it.
    """
    out = b""
    terminated = False
    for p in parts:
        kind = p[0]
        if kind == "label":
            lab = p[1]; out += bytes([len(lab) & 0x3F]) + lab
        elif kind == "len":
            out += bytes([p[1] & 0xFF]) + p[2]
        elif kind == "ptr":
            out += struct.pack("!H", 0xC000 | (p[1] & 0x3FFF)); terminated = True
        elif kind == "raw":
            out += p[1]
        else:
            raise ValueError(f"bad part {kind!r}")
    if not terminated:
        out += b"\x00"
    return out

# ── Question ────────────────────────────────────────────────────────────────
def build_question(qname_wire: bytes, qtype: int, qclass: int = 1) -> bytes:
    return qname_wire + struct.pack("!HH", qtype & 0xFFFF, qclass & 0xFFFF)

# ── EDNS OPT pseudo-RR (goes in the additional section) ─────────────────────
def edns_option(code: int, data: bytes) -> bytes:
    return struct.pack("!HH", code & 0xFFFF, len(data) & 0xFFFF) + data

def build_opt(*, payload=4096, ext_rcode=0, version=0, do=0, z=0,
              options=b"", rdlen_override=None) -> bytes:
    """OPT RR. rdlen_override lets us lie about RDLENGTH."""
    name = b"\x00"
    ttl = ((ext_rcode & 0xFF) << 24) | ((version & 0xFF) << 16) \
        | ((do & 1) << 15) | (z & 0x7FFF)
    rdlen = len(options) if rdlen_override is None else rdlen_override
    return name + struct.pack("!HHIH", 41, payload & 0xFFFF, ttl & 0xFFFFFFFF,
                              rdlen & 0xFFFF) + options

def ecs_option(*, family=1, src_prefix=24, scope_prefix=0, address=b"\xc0\xa8\x00") -> bytes:
    """EDNS Client Subnet (option code 8). Defaults are well-formed; override
    every field to violate RFC 7871 (family, prefix vs address length, etc.)."""
    body = struct.pack("!HBB", family & 0xFFFF, src_prefix & 0xFF, scope_prefix & 0xFF) + address
    return edns_option(8, body)

# ── One-shot query assembler ────────────────────────────────────────────────
def make_query(qname_wire, qtype, *, qclass=1, opt=None, header_kw=None, extra_ar=b""):
    hk = dict(header_kw or {})
    ar_count = (1 if opt else 0) + (1 if extra_ar else 0)
    hk.setdefault("ar", ar_count)
    hk.setdefault("qd", 1)
    txid = hk.pop("txid", random.randint(0, 0xFFFF))
    pkt = build_header(txid, **hk)
    pkt += build_question(qname_wire, qtype, qclass)
    if opt:
        pkt += opt
    if extra_ar:
        pkt += extra_ar
    return pkt

def hexdump(b: bytes, width=16) -> str:
    lines = []
    for i in range(0, len(b), width):
        chunk = b[i:i+width]
        hexs = " ".join(f"{x:02x}" for x in chunk)
        ascii_ = "".join(chr(x) if 32 <= x < 127 else "." for x in chunk)
        lines.append(f"{i:04x}  {hexs:<{width*3}}  {ascii_}")
    return "\n".join(lines)

# quick smoke test — build a normal A query and show it
_demo = make_query(enc_name("www." + TEST_ZONE), 1, header_kw={"rd": 1})
print(hexdump(_demo))

## 2. Transport + liveness canary + crash-isolation harness

We send over both UDP and TCP. After **every** test packet we fire a canary
query; if the resolver stops answering, we've found (and pinpointed) a
candidate silver bullet.

In [ ]:
def send_udp(pkt, ip, port, timeout=DEFAULT_UDP_TIMEOUT):
    guard_target(ip)
    s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    s.settimeout(timeout)
    try:
        s.sendto(pkt, (ip, port))
        data, _ = s.recvfrom(65535)
        return data
    except socket.timeout:
        return None
    except OSError as e:
        return ("ERR", str(e))
    finally:
        s.close()

def send_tcp(pkt, ip, port, timeout=DEFAULT_TCP_TIMEOUT):
    guard_target(ip)
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(timeout)
    try:
        s.connect((ip, port))
        s.sendall(struct.pack("!H", len(pkt)) + pkt)
        hdr = s.recv(2)
        if len(hdr) < 2:
            return None
        (rlen,) = struct.unpack("!H", hdr)
        buf = b""
        while len(buf) < rlen:
            chunk = s.recv(rlen - len(buf))
            if not chunk:
                break
            buf += chunk
        return buf
    except (socket.timeout, ConnectionError, OSError) as e:
        return ("ERR", str(e)) if isinstance(e, OSError) and not isinstance(e, socket.timeout) else None
    finally:
        s.close()

SEND = {"udp": send_udp, "tcp": send_tcp}

def resolver_alive(retries=3, base_timeout=2.0):
    """Canary: a plain RD=1 A query for CANARY_NAME. Any DNS response (even
    SERVFAIL/REFUSED) means the resolver process is still serving."""
    for i in range(retries):
        pkt = make_query(enc_name(CANARY_NAME), 1, header_kw={"rd": 1})
        r = send_udp(pkt, RESOLVER_IP, RESOLVER_PORT, timeout=base_timeout + i)
        if isinstance(r, bytes) and len(r) >= 12:
            return True
        time.sleep(0.4)
    return False

@dataclass
class Campaign:
    name: str
    transport: str = "udp"
    check_every: int = 1          # canary after every N packets (1 = pinpoint)
    stop_on_crash: bool = True
    settle: float = 0.0           # sleep between packets (rate limit)

def run_campaign(camp: Campaign, packets):
    """packets: iterable of (label:str, pkt:bytes). Returns list[dict]."""
    sender = SEND[camp.transport]
    rows = []
    if not resolver_alive():
        raise RuntimeError("Resolver is not answering the canary BEFORE we start. "
                           "Check RESOLVER_IP/PORT and that it can resolve CANARY_NAME.")
    n = 0
    for label, pkt in packets:
        n += 1
        t0 = time.perf_counter()
        resp = sender(pkt, RESOLVER_IP, RESOLVER_PORT)
        dt = (time.perf_counter() - t0) * 1000.0

        if isinstance(resp, tuple) and resp[0] == "ERR":
            rcode, kind, rlen = None, f"oserror:{resp[1]}", 0
        elif resp is None:
            rcode, kind, rlen = None, "timeout", 0
        else:
            rlen = len(resp)
            rcode = (struct.unpack("!H", resp[2:4])[0] & 0xF) if rlen >= 4 else None
            kind = "answer"

        alive = True
        if (n % camp.check_every) == 0 or resp is None:
            alive = resolver_alive()

        rows.append({
            "campaign": camp.name, "seq": n, "label": label,
            "transport": camp.transport, "pkt_len": len(pkt),
            "latency_ms": round(dt, 2), "resp_len": rlen,
            "rcode": rcode, "result": kind, "resolver_alive": alive,
            "pkt_hex": pkt.hex(),
        })

        if not alive:
            print(f"  🚨 RESOLVER WENT DARK after #{n} [{label}]  (last resp={kind})")
            print("     packet hex:", pkt.hex())
            if camp.stop_on_crash:
                print("     stopping campaign (stop_on_crash=True). Restart your resolver, "
                      "then re-run from the next index to continue.")
                break
        if camp.settle:
            time.sleep(camp.settle)
    return rows

def summarize(rows):
    import pandas as pd
    df = pd.DataFrame(rows)
    if df.empty:
        print("no rows"); return df
    dead = df[~df["resolver_alive"]]
    print(f"{len(df)} packets sent | timeouts={sum(df.result=='timeout')} | "
          f"oserrors={sum(df.result.str.startswith('oserror'))} | "
          f"resolver-down events={len(dead)}")
    if not dead.empty:
        print("\n⚠️  Candidate silver-bullet packets (resolver stopped answering):")
        for _, r in dead.iterrows():
            print(f"   [{r.campaign}] #{r.seq} {r.label}  hex={r.pkt_hex}")
    return df

## 3. Permutation generators

Each generator yields `(label, packet_bytes)`. Labels are human-readable so a
crash points straight at the offending construction. Knobs at the top of each
keep default runs bounded; flip `EXHAUSTIVE=True` to sweep full ranges.

In [ ]:
EXHAUSTIVE = False   # True = full 0..65535 sweeps etc. (slow, tens of thousands of packets)

Q = TEST_ZONE  # shorthand; queries land in our controllable zone

# ---- 3a. QTYPE sweep ------------------------------------------------------
NAMED_QTYPES = {
    "A":1,"NS":2,"CNAME":5,"SOA":6,"NULL":10,"PTR":12,"HINFO":13,"MX":15,"TXT":16,
    "RP":17,"AFSDB":18,"SIG":24,"KEY":25,"AAAA":28,"LOC":29,"SRV":33,"NAPTR":35,
    "KX":36,"CERT":37,"DNAME":39,"OPT":41,"APL":42,"DS":43,"SSHFP":44,"IPSECKEY":45,
    "RRSIG":46,"NSEC":47,"DNSKEY":48,"DHCID":49,"NSEC3":50,"NSEC3PARAM":51,
    "TLSA":52,"SMIMEA":53,"HIP":55,"CDS":59,"CDNSKEY":60,"OPENPGPKEY":61,"CSYNC":62,
    "ZONEMD":63,"SVCB":64,"HTTPS":65,"SPF":99,"EUI48":108,"EUI64":109,"TKEY":249,
    "TSIG":250,"IXFR":251,"AXFR":252,"MAILB":253,"ANY":255,"URI":256,"CAA":257,
    "AVC":258,"DOA":259,"TA":32768,"DLV":32769,
    "TYPE0":0,"RESERVED65535":65535,"UNASSIGNED_666":666,"META_128":128,
}
def gen_qtypes():
    if EXHAUSTIVE:
        seen = set(NAMED_QTYPES.values())
        for name, t in NAMED_QTYPES.items():
            yield (f"qtype/{name}", make_query(enc_name("a." + Q), t))
        for t in range(0, 65536):
            if t in seen:
                continue
            yield (f"qtype/{t}", make_query(enc_name("a." + Q), t))
    else:
        for name, t in NAMED_QTYPES.items():
            yield (f"qtype/{name}", make_query(enc_name("a." + Q), t))

# ---- 3b. Header flags / opcodes / rcodes / Z-bit --------------------------
def gen_header_bits():
    base = enc_name("a." + Q)
    for op in range(0, 16):                      # opcodes incl. reserved
        yield (f"opcode/{op}", make_query(base, 1, header_kw={"opcode": op, "rd": 1}))
    for rc in range(0, 16):                       # rcode set in a *query*
        yield (f"qrcode/{rc}", make_query(base, 1, header_kw={"rcode": rc, "rd": 1}))
    # every single-bit flag, plus the reserved Z bit and QR=1 (response as query)
    for bit in ["qr","aa","tc","rd","ra","z","ad","cd"]:
        yield (f"flag/{bit}", make_query(base, 1, header_kw={bit: 1}))
    # all flag bits set at once
    yield ("flag/all", make_query(base, 1, header_kw=dict(qr=1,aa=1,tc=1,rd=1,ra=1,z=1,ad=1,cd=1)))
    # QDCOUNT lies: claim questions we didn't include / include more than we claim
    yield ("count/qd0-butq", make_query(base, 1, header_kw={"qd": 0}))
    yield ("count/qd5-but1", make_query(base, 1, header_kw={"qd": 5}))
    # multiple real questions (QDCOUNT>1)
    twoq = build_header(random.randint(0,0xFFFF), qd=2, rd=1) \
         + build_question(enc_name("a." + Q), 1) + build_question(enc_name("b." + Q), 28)
    yield ("count/qd2-real", ("mq", twoq)[1])
    # lie about AN/NS/AR without providing records
    for sec, kw in [("an",{"an":3}),("ns",{"ns":3}),("ar",{"ar":3})]:
        yield (f"count/{sec}3-empty", make_query(base, 1, header_kw=kw))

# ---- 3c. QNAME boundary abuse --------------------------------------------
def gen_qname_boundaries():
    # max total wire length ~255. Build near/over the edge.
    lab = b"a" * 63
    # 3 x 63 + tail = 255-ish (valid), then push over
    valid_labels = [("len", 63, lab)] * 3 + [("len", 60, b"b"*60)]
    yield ("qname/near255", make_query(raw_name(valid_labels), 1))
    over = [("len", 63, lab)] * 4 + [("len", 63, lab)]   # ~320 octets, illegal
    yield ("qname/over255", make_query(raw_name(over), 1))
    # label length boundary
    yield ("qname/label63", make_query(raw_name([("len",63,lab),("label",b"x")]), 1))
    yield ("qname/label64-illegal", make_query(raw_name([("len",64,b"a"*64),("label",b"x")]), 1))
    yield ("qname/label255-illegal", make_query(raw_name([("len",255,b"a"*255),("label",b"x")]), 1))
    # depth: many tiny labels
    deep_ok  = [("label", b"a")] * 100
    deep_big = [("label", b"a")] * 200
    yield ("qname/depth100", make_query(raw_name(deep_ok), 1))
    yield ("qname/depth200", make_query(raw_name(deep_big), 1))
    # empty label in the middle (double dot) and a zero-length non-terminal
    yield ("qname/empty-mid", make_query(raw_name([("label",b"a"),("len",0,b""),("label",b"b")]), 1))
    # non-printable / control / high bytes in labels
    yield ("qname/nullbyte",  make_query(raw_name([("len",3,b"a\x00b")]), 1))
    yield ("qname/ctrl",      make_query(raw_name([("len",4,b"\x01\x02\x03\x04")]), 1))
    yield ("qname/highbytes", make_query(raw_name([("len",4,b"\xff\xfe\xfd\xfc")]), 1))
    # compression pointer inside a QUESTION qname (illegal) — self loop & wild
    yield ("qname/ptr-self",  make_query(raw_name([("ptr", 12)]), 1))   # points at header/qname start
    yield ("qname/ptr-wild",  make_query(raw_name([("ptr", 0x3FFF)]), 1))
    yield ("qname/label-then-ptr", make_query(raw_name([("label",b"a"),("ptr",12)]), 1))
    # trailing garbage after a valid qname+question
    good = make_query(enc_name("a." + Q), 1)
    yield ("qname/trailing-junk", ("t", good + b"\xde\xad\xbe\xef" * 8)[1])
    # truncated packet (cut mid-question)
    yield ("qname/truncated", ("t", good[:len(good)-3])[1])
    # empty root as qname
    yield ("qname/root", make_query(b"\x00", 1))

# ---- 3d. EDNS0 permutations ----------------------------------------------
def gen_edns():
    base = enc_name("a." + Q)
    # payload size boundaries
    for ps in [0, 1, 511, 512, 1220, 1232, 4096, 65535]:
        yield (f"edns/payload{ps}", make_query(base, 1, opt=build_opt(payload=ps), header_kw={"ar":1}))
    # version: 0 valid, others should yield BADVERS but must not crash
    for v in [0, 1, 15, 255]:
        yield (f"edns/version{v}", make_query(base, 1, opt=build_opt(version=v), header_kw={"ar":1}))
    # DO bit, extended rcode, reserved Z bits
    yield ("edns/do", make_query(base, 1, opt=build_opt(do=1), header_kw={"ar":1}))
    yield ("edns/extrcode", make_query(base, 1, opt=build_opt(ext_rcode=0xFF), header_kw={"ar":1}))
    yield ("edns/zbits", make_query(base, 1, opt=build_opt(z=0x7FFF), header_kw={"ar":1}))
    # RDLENGTH lie: claim big rdata, send none
    yield ("edns/rdlen-lie", make_query(base, 1, opt=build_opt(options=b"", rdlen_override=100), header_kw={"ar":1}))
    # option-code sweep (known + unknown), well-formed empty payloads
    KNOWN_OPTS = {1:"LLQ",2:"UL",3:"NSID",5:"DAU",6:"DHU",7:"N3U",8:"ECS",9:"EXPIRE",
                  10:"COOKIE",11:"TCPKEEPALIVE",12:"PADDING",13:"CHAIN",14:"KEYTAG",
                  15:"EDE",16:"CLIENTTAG",17:"SERVERTAG",18:"REPORTCHANNEL",20292:"UMBRELLA"}
    for codev, nm in KNOWN_OPTS.items():
        yield (f"edns/opt-{nm}", make_query(base, 1, opt=build_opt(options=edns_option(codev, b"")), header_kw={"ar":1}))
    # malformed option: declared length longer than actual payload
    bad_opt = struct.pack("!HH", 3, 50) + b"\x00\x01"   # NSID says 50, gives 2
    yield ("edns/opt-badlen", make_query(base, 1, opt=build_opt(options=bad_opt), header_kw={"ar":1}))
    # cookie with wrong sizes (RFC 7873: client=8, full=16..40)
    yield ("edns/cookie-short", make_query(base, 1, opt=build_opt(options=edns_option(10, b"\x01\x02\x03")), header_kw={"ar":1}))
    yield ("edns/cookie-huge",  make_query(base, 1, opt=build_opt(options=edns_option(10, b"\xaa"*64)), header_kw={"ar":1}))
    # padding: enormous
    yield ("edns/padding-huge", make_query(base, 1, opt=build_opt(options=edns_option(12, b"\x00"*1400)), header_kw={"ar":1}))
    # many options stacked
    stacked = b"".join(edns_option(c, b"") for c in range(3, 40))
    yield ("edns/opt-stack", make_query(base, 1, opt=build_opt(options=stacked), header_kw={"ar":1}))
    # two OPT RRs (illegal: only one allowed)
    two_opt = build_opt() + build_opt()
    pkt = build_header(random.randint(0,0xFFFF), rd=1, qd=1, ar=2) + build_question(base,1) + two_opt
    yield ("edns/two-opt", ("t", pkt)[1])

# ---- 3e. EDNS Client Subnet abuse (RFC 7871) ------------------------------
def gen_ecs():
    base = enc_name("a." + Q)
    def q(nm, **kw):
        return (f"ecs/{nm}", make_query(base, 1, opt=build_opt(options=ecs_option(**kw)), header_kw={"ar":1}))
    yield q("v4-normal", family=1, src_prefix=24, address=b"\xc0\xa8\x00")
    yield q("v4-src32", family=1, src_prefix=32, address=b"\xc0\xa8\x00\x01")
    yield q("v4-src33-illegal", family=1, src_prefix=33, address=b"\xc0\xa8\x00\x01")
    yield q("v4-src0-noaddr", family=1, src_prefix=0, address=b"")
    yield q("v4-addr-too-short", family=1, src_prefix=32, address=b"\xc0")      # prefix wants 4 bytes
    yield q("v4-addr-too-long", family=1, src_prefix=8, address=b"\x0a"*16)     # prefix wants 1 byte
    yield q("v6-src128", family=2, src_prefix=128, address=b"\x20\x01" + b"\x00"*14)
    yield q("v6-src129-illegal", family=2, src_prefix=129, address=b"\x20\x01" + b"\x00"*15)
    yield q("family0", family=0, src_prefix=24, address=b"\xc0\xa8\x00")
    yield q("family3-unknown", family=3, src_prefix=24, address=b"\xc0\xa8\x00")
    yield q("family65535", family=65535, src_prefix=255, address=b"\xff"*8)
    yield q("scope-nonzero-in-query", family=1, src_prefix=24, scope_prefix=24, address=b"\xc0\xa8\x00")

# ---- 3f. Oversized / junk / random-fuzz ----------------------------------
def gen_junk(count=200, seed=1337):
    rnd = random.Random(seed)
    base = enc_name("a." + Q)
    # totally random payloads
    for i in range(count):
        n = rnd.randint(0, 600)
        yield (f"junk/random{i}", bytes(rnd.randint(0,255) for _ in range(n)))
    # valid header, random tail
    for i in range(count // 4):
        pkt = build_header(rnd.randint(0,0xFFFF), rd=1, qd=1) + build_question(base,1)
        pkt += bytes(rnd.randint(0,255) for _ in range(rnd.randint(0, 400)))
        yield (f"junk/hdr+rand{i}", pkt)
    # bit-flip a valid query
    good = bytearray(make_query(base, 1, opt=build_opt(do=1), header_kw={"ar":1}))
    for i in range(len(good)):
        m = bytearray(good); m[i] ^= 0xFF
        yield (f"junk/flip@{i}", bytes(m))

# registry so campaigns are easy to run
GENERATORS = {
    "qtypes": gen_qtypes,
    "header": gen_header_bits,
    "qname":  gen_qname_boundaries,
    "edns":   gen_edns,
    "ecs":    gen_ecs,
    "junk":   gen_junk,
}
# show counts (non-exhaustive)
for k, g in GENERATORS.items():
    try:
        print(f"{k:8s}: {sum(1 for _ in g())} packets")
    except Exception as e:
        print(f"{k:8s}: <lazy/param> ({e})")

### 3g. Offline self-test (no resolver required)

Before pointing at a real resolver, confirm the harness plumbing works by
running one campaign against the built-in malicious auth server (Part B) or just
inspecting a few crafted packets. This cell only renders hexdumps.

In [ ]:
for label, pkt in list(gen_qname_boundaries())[:4]:
    print(f"### {label}  ({len(pkt)} bytes)")
    print(hexdump(pkt))
    print()

## Part A — run the query-side fuzz campaigns

Point `RESOLVER_IP/PORT` at your resolver (Cell 0). Each campaign canary-checks
after every packet so a crash is pinpointed to the exact construction. Results
land in a DataFrame you can sort/export.

> If your resolver auto-restarts on crash, set `stop_on_crash=False` to keep
> going and collect *all* killers in one pass.

In [ ]:
import pandas as pd

ALL_ROWS = []

def run(name, transport="udp", check_every=1, stop_on_crash=True, settle=0.0, gen=None, **genkw):
    g = (gen or GENERATORS[name])(**genkw) if genkw else (gen or GENERATORS[name])()
    camp = Campaign(name=name, transport=transport, check_every=check_every,
                    stop_on_crash=stop_on_crash, settle=settle)
    print(f"▶ campaign {name!r} over {transport} …")
    rows = run_campaign(camp, g)
    ALL_ROWS.extend(rows)
    return summarize(rows)

# --- uncomment to run against your resolver ---
# run("qtypes")
# run("header")
# run("qname")
# run("edns")
# run("ecs")
# run("junk")
# run("qtypes", transport="tcp")
# run("qname",  transport="tcp")
print("Ready. Uncomment the run(...) calls above once your resolver is up.")

In [ ]:
# Consolidated results + candidate silver bullets
if ALL_ROWS:
    df = pd.DataFrame(ALL_ROWS)
    display(df.groupby(["campaign","result"]).size().unstack(fill_value=0))
    kills = df[~df["resolver_alive"]]
    if not kills.empty:
        print("\n🎯 SILVER-BULLET CANDIDATES")
        display(kills[["campaign","seq","label","transport","pkt_len","result","pkt_hex"]])
    else:
        print("\nNo resolver-down events recorded. 🎉 (or resolver auto-recovered)")
    df.to_csv("/workspace/fuzz-results.csv", index=False)
    print("saved -> /workspace/fuzz-results.csv")
else:
    print("No campaigns have been run yet.")

## Part B — Malicious authoritative server (RFC-violating responses)

A pure-Python UDP+TCP server. It runs in a background thread inside this kernel.
For any query, it looks at the **first label** of the QNAME to pick a *violation
handler*; e.g. `ptrloop.fuzz.test` triggers a compression-pointer loop in the
response, `countlie.fuzz.test` returns a header that claims records it didn't
send, and so on.

Point your resolver at this server as the authoritative for `TEST_ZONE`. The
next cell prints ready-to-paste BIND/Unbound/Knot config.

In [ ]:
# ── Low-level response builders (share the encoder from Cell 1) ─────────────
def _copy_question(query: bytes):
    """Return (txid, question_wire, qname_end_offset). Assumes 1 question."""
    txid = query[0:2]
    off = 12
    # walk the qname
    start = off
    while True:
        ln = query[off]
        if ln == 0:
            off += 1; break
        if ln & 0xC0 == 0xC0:
            off += 2; break
        off += 1 + ln
    off += 4  # qtype+qclass
    return txid, query[12:off], off

def _resp_header(txid, *, aa=1, ra=1, rcode=0, an=1, ns=0, ar=0, tc=0):
    flags = (1<<15) | (aa<<10) | (tc<<9) | (1<<8) | (ra<<7) | (rcode & 0xF)
    return txid + struct.pack("!HHHHH", flags, 1, an, ns, ar)

def _a_rr(name_wire, ip="10.9.9.9", ttl=300):
    return name_wire + struct.pack("!HHIH", 1, 1, ttl, 4) + socket.inet_aton(ip)

def _name_ptr(off=12):
    return struct.pack("!H", 0xC000 | off)

# ── Violation handlers: (query_bytes, qname_wire, qtype) -> response_bytes ──
def h_valid(q, qn, qt):
    txid, ques, _ = _copy_question(q)
    return _resp_header(txid, an=1) + ques + _a_rr(_name_ptr(12))

def h_countlie(q, qn, qt):
    # header claims 100 answers, we send one
    txid, ques, _ = _copy_question(q)
    return _resp_header(txid, an=100) + ques + _a_rr(_name_ptr(12))

def h_countlie_zero(q, qn, qt):
    # header claims 0 answers but we append records after it
    txid, ques, _ = _copy_question(q)
    return _resp_header(txid, an=0) + ques + _a_rr(_name_ptr(12))

def h_ptrloop(q, qn, qt):
    # answer owner name is a compression pointer that points at itself
    txid, ques, off = _copy_question(q)
    hdr = _resp_header(txid, an=1) + ques
    self_off = len(hdr)                    # offset where the RR starts
    rr = struct.pack("!H", 0xC000 | self_off) + struct.pack("!HHIH", 1,1,300,4) + socket.inet_aton("10.0.0.1")
    return hdr + rr

def h_badptr(q, qn, qt):
    # pointer to an offset past the end of the packet
    txid, ques, _ = _copy_question(q)
    hdr = _resp_header(txid, an=1) + ques
    rr = struct.pack("!H", 0xC000 | 0x0FFF) + struct.pack("!HHIH", 1,1,300,4) + socket.inet_aton("10.0.0.2")
    return hdr + rr

def h_longlabel(q, qn, qt):
    # owner name with an illegal 200-byte label
    txid, ques, _ = _copy_question(q)
    bad = bytes([200]) + b"a"*200 + b"\x00"
    rr = bad + struct.pack("!HHIH", 1,1,300,4) + socket.inet_aton("10.0.0.3")
    return _resp_header(txid, an=1) + ques + rr

def h_badrdlen(q, qn, qt):
    # RDLENGTH says 40 but only 4 bytes of rdata follow
    txid, ques, _ = _copy_question(q)
    rr = _name_ptr(12) + struct.pack("!HHIH", 1,1,300,40) + socket.inet_aton("10.0.0.4")
    return _resp_header(txid, an=1) + ques + rr

def h_cnameloop(q, qn, qt):
    # a -> b and b -> a within the same response
    txid, ques, _ = _copy_question(q)
    hdr = _resp_header(txid, an=2) + ques
    a_name = _name_ptr(12)
    b_name = enc_name("b." + TEST_ZONE)
    # CNAME a -> b
    rr1 = a_name + struct.pack("!HHIH", 5,1,300,len(b_name)) + b_name
    # CNAME b -> a  (rdata points back with a compression pointer to owner @12)
    back = _name_ptr(12)
    rr2 = b_name + struct.pack("!HHIH", 5,1,300,len(back)) + back
    return hdr + rr1 + rr2

def h_cnamechain(q, qn, qt, depth=120):
    # very long CNAME chain in one message
    txid, ques, _ = _copy_question(q)
    rrs = b""; n = 0
    owner = _name_ptr(12)
    for i in range(depth):
        tgt = enc_name(f"c{i}." + TEST_ZONE)
        rrs += owner + struct.pack("!HHIH", 5,1,300,len(tgt)) + tgt
        owner = tgt; n += 1
    return _resp_header(txid, an=n) + ques + rrs

def h_hugerrset(q, qn, qt, n=3000):
    # thousands of A records (memory / amplification). Sent over TCP-safe path.
    txid, ques, _ = _copy_question(q)
    rrs = b"".join(_name_ptr(12) + struct.pack("!HHIH",1,1,300,4) + struct.pack("!BBBB",10,(i>>16)&255,(i>>8)&255,i&255) for i in range(n))
    return _resp_header(txid, an=n) + ques + rrs

def h_tc_fullanswer(q, qn, qt):
    # TC bit set but a full answer is present (should the resolver retry TCP?)
    txid, ques, _ = _copy_question(q)
    return _resp_header(txid, an=1, tc=1) + ques + _a_rr(_name_ptr(12))

def h_extrcode(q, qn, qt):
    # OPT in the response with an extended rcode + garbage options
    txid, ques, _ = _copy_question(q)
    opt = build_opt(ext_rcode=0xFF, version=1, options=edns_option(3, b"\xff"*10))
    return _resp_header(txid, an=1, ar=1) + ques + _a_rr(_name_ptr(12)) + opt

def h_manyquestions(q, qn, qt):
    # response echoes QDCOUNT=5 but includes one question
    txid, ques, _ = _copy_question(q)
    h = txid + struct.pack("!HHHHH", (1<<15)|(1<<8)|(1<<7), 5, 1, 0, 0)
    return h + ques + _a_rr(_name_ptr(12))

def h_truncated_wire(q, qn, qt):
    # a well-formed-looking header promising an answer, then a truncated RR
    txid, ques, _ = _copy_question(q)
    full = _resp_header(txid, an=1) + ques + _a_rr(_name_ptr(12))
    return full[:len(full)-3]

def h_hugettl(q, qn, qt):
    txid, ques, _ = _copy_question(q)
    rr = _name_ptr(12) + struct.pack("!HHIH", 1,1,0xFFFFFFFF,4) + socket.inet_aton("10.0.0.9")
    return _resp_header(txid, an=1) + ques + rr

def h_canary(q, qn, qt):
    # clean answer so the liveness canary through the resolver works
    return h_valid(q, qn, qt)

VIOLATIONS = {
    "canary":     h_canary,
    "valid":      h_valid,
    "countlie":   h_countlie,
    "countzero":  h_countlie_zero,
    "ptrloop":    h_ptrloop,
    "badptr":     h_badptr,
    "longlabel":  h_longlabel,
    "badrdlen":   h_badrdlen,
    "cnameloop":  h_cnameloop,
    "cnamechain": h_cnamechain,
    "hugerrset":  h_hugerrset,
    "tcfull":     h_tc_fullanswer,
    "extrcode":   h_extrcode,
    "manyq":      h_manyquestions,
    "trunc":      h_truncated_wire,
    "hugettl":    h_hugettl,
}
print("violation handlers:", ", ".join(sorted(VIOLATIONS)))
print(f"\nQuery e.g.  ptrloop.{TEST_ZONE} through the resolver to trigger a handler.")

In [ ]:
# ── The server itself (threaded UDP + TCP) ──────────────────────────────────
class MaliciousAuth:
    def __init__(self, bind_ip, port):
        self.bind_ip, self.port = bind_ip, port
        self._stop = threading.Event()
        self._threads = []
        self.log = []            # (proto, first_label, qtype)
        self.max_log = 500

    def _first_label(self, query):
        # decode the first label of the qname to route on
        try:
            ln = query[12]
            if ln == 0 or ln & 0xC0:
                return ""
            return query[13:13+ln].decode("latin-1").lower()
        except Exception:
            return ""

    def _qtype(self, query):
        try:
            off = 12
            while True:
                ln = query[off]
                if ln == 0: off += 1; break
                if ln & 0xC0: off += 2; break
                off += 1 + ln
            return struct.unpack("!H", query[off:off+2])[0]
        except Exception:
            return 0

    def _respond(self, query, proto):
        first = self._first_label(query)
        qt = self._qtype(query)
        if len(self.log) < self.max_log:
            self.log.append((proto, first, qt))
        handler = VIOLATIONS.get(first, h_valid)
        try:
            qn_end = 12
            return handler(query, query[12:qn_end], qt)
        except Exception as e:
            # never let the auth server die; fall back to a valid answer
            return h_valid(query, b"", qt)

    def _udp_loop(self):
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        s.bind((self.bind_ip, self.port)); s.settimeout(0.5)
        while not self._stop.is_set():
            try:
                data, addr = s.recvfrom(65535)
            except socket.timeout:
                continue
            except OSError:
                break
            if len(data) < 12:
                continue
            try:
                resp = self._respond(data, "udp")
                if resp:
                    s.sendto(resp, addr)
            except Exception:
                pass
        s.close()

    def _tcp_loop(self):
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        s.bind((self.bind_ip, self.port)); s.listen(16); s.settimeout(0.5)
        while not self._stop.is_set():
            try:
                conn, addr = s.accept()
            except socket.timeout:
                continue
            except OSError:
                break
            threading.Thread(target=self._tcp_conn, args=(conn,), daemon=True).start()
        s.close()

    def _tcp_conn(self, conn):
        conn.settimeout(2.0)
        try:
            hdr = conn.recv(2)
            if len(hdr) < 2: return
            (ln,) = struct.unpack("!H", hdr)
            buf = b""
            while len(buf) < ln:
                c = conn.recv(ln - len(buf))
                if not c: break
                buf += c
            if len(buf) >= 12:
                resp = self._respond(buf, "tcp")
                if resp:
                    conn.sendall(struct.pack("!H", len(resp)) + resp)
        except Exception:
            pass
        finally:
            conn.close()

    def start(self):
        self._stop.clear()
        for target in (self._udp_loop, self._tcp_loop):
            t = threading.Thread(target=target, daemon=True)
            t.start(); self._threads.append(t)
        time.sleep(0.3)
        print(f"✅ MaliciousAuth listening on {self.bind_ip}:{self.port} (UDP+TCP)")

    def stop(self):
        self._stop.set()
        time.sleep(0.7)
        self._threads = []
        print("🛑 MaliciousAuth stopped")

AUTH = MaliciousAuth(AUTH_BIND_IP, AUTH_PORT)
AUTH.start()

In [ ]:
# Self-test the auth server locally (no resolver needed): send it a crafted
# query for each violation and show what comes back. Proves the byte-level
# response builders work before you wire in a real resolver.
def _probe_auth(label):
    qn = enc_name(f"{label}.{TEST_ZONE}")
    pkt = make_query(qn, 1, header_kw={"rd": 1})
    r = send_udp(pkt, "127.0.0.1", AUTH_PORT, timeout=2.0)
    return r

for name in ["valid","countlie","ptrloop","badptr","longlabel","badrdlen","cnameloop","trunc","hugettl"]:
    r = _probe_auth(name)
    if isinstance(r, bytes):
        an = struct.unpack("!H", r[6:8])[0]
        print(f"{name:11s} -> {len(r):4d} bytes, ANCOUNT={an}")
    else:
        print(f"{name:11s} -> {r}")
print("\n(hugerrset/cnamechain are large; query them over TCP through the resolver in Part C)")

### Wiring your resolver to the malicious auth server

Configure your recursive resolver to treat `TEST_ZONE` as delegated to this
server (so it queries us for those names) **and** to not fall back to the
internet. Examples below — adjust IP/port to `AUTH_ADVERTISED_IP:AUTH_PORT`.

In [ ]:
ip, port, zone = AUTH_ADVERTISED_IP, AUTH_PORT, TEST_ZONE.rstrip(".")

print("── Unbound (stub-zone forces queries for the zone to us) ──")
print(f"""server:
    do-not-query-localhost: no      # allow talking to 127.0.0.1 in the lab
    harden-glue: no
stub-zone:
    name: "{zone}"
    stub-addr: {ip}@{port}
    stub-no-cache: yes
""")

print("── BIND (static-stub / forward-only to us) ──")
print(f"""zone "{zone}" {{
    type static-stub;
    server-addresses {{ {ip}; }};   // note: BIND uses port 53 unless built to override
}};
// or, simplest for a lab, forward the whole zone:
zone "{zone}" {{
    type forward;
    forward only;
    forwarders {{ {ip} port {port}; }};
}};
""")

print("── Knot Resolver (kresd) ──")
print(f"""policy.add(policy.suffix(
    policy.STUB('{ip}@{port}'),
    {{ todname('{zone}') }}))
""")

print(f"Then verify from your stub:  dig @{RESOLVER_IP} -p {RESOLVER_PORT} canary.{zone} A")

## Part C — End-to-end hunt (resolver parses malicious upstream responses)

Now drive normal queries **through the resolver** for names that trigger each
violation handler. The resolver fetches our RFC-violating response from Part B,
parses it, and (hopefully for it, not for us) survives. We canary-check after
each.

`CANARY_NAME` = `canary.fuzz.test` resolves cleanly via our `h_canary` handler,
so the liveness probe works end-to-end.

In [ ]:
def hunt_upstream(stop_on_crash=True, transport="udp", extra_qtypes=(1,28,255,64,65,46,48)):
    """Query each violation name (several qtypes) through the resolver, watching health."""
    rows = []
    if not resolver_alive():
        raise RuntimeError("resolver canary failed before hunt — check wiring / config")
    targets = [n for n in VIOLATIONS if n not in ("valid","canary")]
    n = 0
    for vio in targets:
        for qt in extra_qtypes:
            n += 1
            qname = enc_name(f"{vio}.{TEST_ZONE}")
            pkt = make_query(qname, qt, header_kw={"rd": 1}, opt=build_opt(do=1), )
            # ^ RD=1 forces recursion; DO=1 exercises the DNSSEC path too
            t0 = time.perf_counter()
            resp = SEND[transport](pkt, RESOLVER_IP, RESOLVER_PORT,
                                   timeout=6.0)   # allow for upstream fetch
            dt = (time.perf_counter()-t0)*1000
            kind = "answer" if isinstance(resp, bytes) else ("timeout" if resp is None else "oserr")
            alive = resolver_alive()
            rows.append({"violation": vio, "qtype": qt, "transport": transport,
                         "latency_ms": round(dt,2),
                         "resp_len": len(resp) if isinstance(resp,bytes) else 0,
                         "result": kind, "resolver_alive": alive})
            if not alive:
                print(f"  🚨 resolver DOWN after {vio}/{qt} ({transport})")
                if stop_on_crash:
                    return rows
    return rows

# --- run it (uncomment once the resolver is wired to Part B) ---
# up_rows = hunt_upstream(transport="udp")
# up_rows += hunt_upstream(transport="tcp")   # TCP path exercises hugerrset/cnamechain
# import pandas as pd
# updf = pd.DataFrame(up_rows); display(updf)
# display(updf[~updf.resolver_alive])
print("Ready. Wire the resolver to Part B, then uncomment hunt_upstream().")
print("Auth server saw these queries so far:", AUTH.log[-10:])

## 4. Report / export

Consolidate Part A + Part C findings and export. Use **File → Save and Export
Notebook As → WebPDF** (offline Chromium is baked into the image) for a
shareable report.

In [ ]:
import pandas as pd
frames = []
if ALL_ROWS:
    frames.append(pd.DataFrame(ALL_ROWS).assign(phase="A-query-fuzz"))
try:
    frames.append(pd.DataFrame(up_rows).assign(phase="C-upstream"))  # noqa: F821
except NameError:
    pass

if frames:
    report = pd.concat(frames, ignore_index=True, sort=False)
    kills = report[~report["resolver_alive"]]
    print(f"Total probes: {len(report)} | resolver-down events: {len(kills)}")
    if not kills.empty:
        print("\n🎯 CONFIRMED SILVER BULLETS")
        cols = [c for c in ["phase","campaign","violation","label","qtype","transport","pkt_hex"] if c in kills.columns]
        display(kills[cols])
    report.to_csv("/workspace/torture-report.csv", index=False)
    print("saved -> /workspace/torture-report.csv")
else:
    print("Nothing to report yet — run Part A and/or Part C first.")

In [ ]:
# Tidy shutdown of the malicious auth server when you're done.
# AUTH.stop()
print("When finished: AUTH.stop()")